## 20. Cross-Dataset Summary and Updated Conclusions

### Skip Connection Effect: Cross-Dataset Matrix

| Architecture | M4-Yearly (H=6) | Tourism-Yearly (H=4) | Weather-96 (H=96) |
|---|---|---|---|
| **TrendWaveletAE** | No effect (ns) | **NO-SKIP better** (p=0.001) | No effect (ns) |
| **TrendWaveletAELG** | No effect (ns) | No effect (ns) | No effect (ns) |
| **GenericAE** | No effect (ns) | **NO-SKIP better** (p=0.011) | No effect (ns) |
| **GenericAELG** | **SKIP rescues** at depth>=20 | Skip marginally helps (ns) | No effect (ns) |
| **Generic** | No effect (ns) | **SKIP helps** (p=0.014) | No effect (ns) |
| **TrendVAE+WaveletV3VAE** | Skip reduces severity (ns) | Skip reduces severity (p=0.010) | No effect (ns) |
| **TrendVAE2+WaveletV3VAE2** | Skip reduces severity (ns) | **Skip reduces severity** (p=0.003) | **NO-SKIP better** (p=0.013) |
| **TrendVAE+HaarWaveletV3** | Skip reduces severity (ns) | No effect (ns) | **NO-SKIP better** (p=0.034) |

### Key Cross-Dataset Findings

1. **Skip connections are NEVER the optimal choice for TrendWavelet blocks.** Across all 3 datasets, the TrendWavelet no-skip baseline matches or beats the skip variant. The combined polynomial+DWT basis provides inherent depth stability.

2. **The GenericAELG rescue effect is M4-specific.** The dramatic bimodal collapse (SMAPE 14->36) only fully manifests on M4-Yearly. Tourism shows a milder form (one seed out of three at 30 stacks), and Weather shows no collapse at all. The mechanism likely depends on the interaction between horizon length, data normalization, and the learned gate initialization.

3. **Shallow depth wins on short horizons.** Tourism's best config (GAE10, 10 stacks) and Weather's best SMAPE (TVH10, 10 stacks) are both at minimal depth. M4's best (TW16-24) benefits from moderate depth. This suggests a rough heuristic: `optimal_stacks ~ max(10, H)`.

4. **Double-VAE catastrophe severity scales inversely with forecast horizon.** Tourism (H=4): 7-8x worse. M4 (H=6): 2-3x worse. Weather (H=96): ~1.1x worse. Longer horizons dilute the reparameterization noise across more forecast points.

5. **Fixed alpha=0.1 is the safest default** when skip is used (7/12 wins on Tourism, mixed on Weather). Learnable alpha adds optimization complexity without consistent benefit.

### Updated Skip Connection Decision Tree (All Datasets)

```
Is your architecture depth-stable? (TrendWavelet, GenericAE, Generic on long horizons)
  YES -> Do NOT use skip connections.
  NO  -> Is it GenericAELG on M4-Yearly (or similar short-horizon competition data)?
           YES -> Use skip_distance=floor(n_stacks/6), alpha=0.1
           NO  -> Is it a double-VAE pairing?
                    YES -> Redesign: pair VAE with deterministic block or use deterministic-only.
                    NO  -> Test empirically. Default: OFF.
```

### What to Test Next

1. **GAE10 on Tourism with higher latent_dim.** The skip study used ld=16. The SOTA uses TrendWaveletAELG_coif3_eq_bcast_td3_ld16 (SMAPE 20.864). GAE10 at 20.53 is competitive -- worth testing GAE10 with ld=32 and active_g.

2. **Weather v2 completion.** Only 2 of 9 expected R3 configs completed. The incomplete halving limits confidence in Weather v2 rankings. Re-run with extended budget.

3. **Mixed depth study.** Test architectures with heterogeneous stack depths (e.g., 5 stacks of TrendWav + 5 stacks of GenericAE) to see if combining depth-stable blocks with different inductive biases outperforms homogeneous stacks.

In [ ]:
# Fixed vs learnable alpha comparison across datasets
print("Fixed vs Learnable Alpha Comparison (v1 studies only -- v2 uses only fixed)")
print("="*80)

alpha_results = []
for label, dframe in [('Tourism v1', tourism_v1), ('Weather v1', weather_v1)]:
    best = get_best_round(dframe)
    configs = best['config_name'].unique()
    
    for base in set(c.rsplit('_', 1)[0] for c in configs if 'skip' in c and ('learn' in c or 'a01' in c)):
        fixed_name = base + '_a01'
        learn_name = base + '_learn'
        if fixed_name in configs and learn_name in configs:
            f_vals = best[best['config_name']==fixed_name]['smape'].values
            l_vals = best[best['config_name']==learn_name]['smape'].values
            if len(f_vals) > 0 and len(l_vals) > 0:
                winner = 'Fixed' if np.mean(f_vals) < np.mean(l_vals) else 'Learnable'
                delta = np.mean(l_vals) - np.mean(f_vals)
                alpha_results.append({
                    'Dataset': label, 'Config': base,
                    'Fixed': np.mean(f_vals), 'Learnable': np.mean(l_vals),
                    'Delta': delta, 'Winner': winner,
                })

alpha_df = pd.DataFrame(alpha_results)
print(alpha_df.to_string(index=False, float_format='%.3f'))

# Count wins
tourism_wins = alpha_df[alpha_df['Dataset']=='Tourism v1']
weather_wins = alpha_df[alpha_df['Dataset']=='Weather v1']
print(f"\nTourism: Fixed wins {sum(tourism_wins['Winner']=='Fixed')}/{len(tourism_wins)}")
print(f"Weather: Fixed wins {sum(weather_wins['Winner']=='Fixed')}/{len(weather_wins)}")
print(f"Overall: Fixed wins {sum(alpha_df['Winner']=='Fixed')}/{len(alpha_df)}")
print(f"\nConclusion: Fixed alpha=0.1 wins {sum(alpha_df['Winner']=='Fixed')}/{len(alpha_df)} "
      f"comparisons across Tourism+Weather. "
      f"On Weather, learnable alpha wins more often ({sum(weather_wins['Winner']=='Learnable')}/{len(weather_wins)}) "
      f"-- possibly because Weather's longer training gives the learnable parameter time to optimize.")

## 19. Fixed vs Learnable Alpha: Cross-Dataset

The M4 study found fixed alpha=0.1 beats learnable in 4/5 comparisons. Does this hold on Tourism and Weather?

### Interpretation

**Tourism:** GenericAELG shows the bimodal collapse at 30 stacks (one seed hits 69.9 while others are 27-28), matching the M4 pattern. At 10 stacks, GAELG is universally poor on Tourism (SMAPE 27-38) even without collapse -- GenericAELG's lack of inductive bias makes it ineffective for the very short H=4 forecast.

**Weather:** No clean bimodal collapse. GAELG at all depths (10-30) shows moderate performance (39-51 SMAPE) without the dramatic seed-dependent failures seen on M4. Weather's data normalization and longer sequences may provide enough gradient signal to avoid the sharp loss plateaus that trap M4 seeds.

**GenericAE (v2) confirms depth stability** on both Tourism and Weather. No bimodal failures at any depth, consistent with M4 v2 findings. The learned gate in AERootBlockLG remains the identified source of depth instability.

In [ ]:
# GenericAELG depth degradation across datasets
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (ds_label, dframe) in zip(axes, [('Tourism v1', tourism_v1), ('Weather v1', weather_v1)]):
    r1_data = dframe[dframe['search_round'] == 1]
    gaelg = r1_data[r1_data['architecture'] == 'GenericAELG']
    
    for depth in sorted(gaelg['n_stacks'].unique()):
        sub = gaelg[gaelg['n_stacks'] == depth]
        noskip = sub[~sub['has_skip']]['smape'].values
        
        if len(noskip) > 0:
            # Plot individual seeds as scatter + box
            jitter = np.random.normal(depth, 0.3, len(noskip))
            ax.scatter(jitter, noskip, color='#E91E63', alpha=0.6, s=80, zorder=3, edgecolors='black')
            ax.boxplot([noskip], positions=[depth], widths=1.5, 
                       patch_artist=True, boxprops=dict(facecolor='#E91E63', alpha=0.3),
                       showfliers=False)
            
            # Mark bimodal seeds
            if any(noskip > 40):
                ax.annotate('BIMODAL', (depth, max(noskip)), fontsize=8, color='red',
                            ha='center', va='bottom')
    
    ax.set_xlabel('Number of Stacks')
    ax.set_ylabel('SMAPE')
    ax.set_title(f'{ds_label}: GenericAELG No-Skip (R1, individual seeds)')
    ax.set_xticks(sorted(gaelg['n_stacks'].unique()))
    ax.grid(alpha=0.3)

plt.suptitle('GenericAELG Depth Collapse: Tourism vs Weather', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Also show GenericAE (v2) for comparison
print("\nGenericAE (v2) depth stability (for comparison -- no bimodal expected):")
for label, dframe in [('Tourism v2', tourism_v2), ('Weather v2', weather_v2)]:
    r1 = dframe[dframe['search_round'] == 1]
    gae = r1[r1['architecture'] == 'GenericAE']
    for depth in sorted(gae['n_stacks'].unique()):
        noskip = gae[(gae['n_stacks'] == depth) & (~gae['has_skip'])]['smape']
        if len(noskip) > 0:
            seeds = ', '.join(f'{x:.1f}' for x in noskip.values)
            bimodal = ' **BIMODAL**' if any(noskip > noskip.mean() * 1.5) else ''
            print(f"  {label} GAE{depth}: SMAPE={noskip.mean():.1f} +/-{noskip.std():.1f} [{seeds}]{bimodal}")

## 18. GenericAELG Bimodal Failure: Dataset-Dependent

The M4 study showed GenericAELG collapsing at depth >= 20 with a distinctive bimodal pattern (2/3 seeds stuck at SMAPE ~48, 1/3 converge normally). Does this reproduce on Tourism and Weather?

In [ ]:
# Double-VAE comparison across datasets
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vae_arch_groups = {
    'Double-VAE': ['TrendVAE_WaveletV3VAE'],
    'Double-VAE2': ['TrendVAE2_WaveletV3VAE2'],
    'Single-VAE+Det': ['TrendVAE_HaarWaveletV3'],
    'Deterministic': ['TrendWaveletAE', 'TrendWaveletAELG', 'TrendAELG_WaveletV3AELG'],
}

group_colors = {
    'Double-VAE': '#F44336', 'Double-VAE2': '#795548',
    'Single-VAE+Det': '#9C27B0', 'Deterministic': '#2196F3',
}

for ax, (ds_label, dframe) in zip(axes, 
    [('Tourism v2 (H=4)', tourism_v2), ('M4 v2 (H=6)', df), ('Weather v2 (H=96)', weather_v2)]):
    
    best = get_best_round(dframe)
    
    means = []
    labels = []
    errs = []
    bar_colors = []
    
    for group, archs in vae_arch_groups.items():
        sub = best[best['architecture'].isin(archs)]
        if len(sub) == 0:
            continue
        means.append(sub['smape'].mean())
        errs.append(sub['smape'].std())
        labels.append(group)
        bar_colors.append(group_colors[group])
    
    bars = ax.bar(range(len(means)), means, yerr=errs, capsize=5, 
                  color=bar_colors, edgecolor='white', alpha=0.85)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=9, rotation=15)
    ax.set_ylabel('Mean SMAPE')
    ax.set_title(ds_label)
    
    for i, (m, e) in enumerate(zip(means, errs)):
        ax.text(i, m + e + 0.5, f'{m:.1f}', ha='center', fontsize=10, fontweight='bold')

fig.suptitle('Double-VAE Severity Scales Inversely with Forecast Horizon', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Statistical comparison
print("\nDouble-VAE vs Deterministic (MWU tests):")
print("-" * 70)
for label, dframe in [('Tourism v2', tourism_v2), ('M4 v2', df), ('Weather v2', weather_v2)]:
    best = get_best_round(dframe)
    det = best[best['architecture'].isin(['TrendWaveletAE', 'TrendWaveletAELG'])]['smape'].values
    dvae = best[best['architecture'].isin(['TrendVAE_WaveletV3VAE', 'TrendVAE2_WaveletV3VAE2'])]['smape'].values
    
    if len(det) >= 3 and len(dvae) >= 3:
        u, p = mannwhitneyu(det, dvae)
        ratio = np.mean(dvae) / np.mean(det)
        print(f"  {label:12s}: Det={np.mean(det):.1f}, DVAE={np.mean(dvae):.1f}, "
              f"Ratio={ratio:.1f}x, MWU p={p:.6f}")

## 17. Double-VAE Catastrophe: Tourism vs Weather

The M4 analysis showed double-VAE pairing is catastrophically bad (SMAPE 29-44 vs 14 for deterministic). Does this generalize?

**Tourism:** Double-VAE is even WORSE than on M4. TrendVAE+WaveletV3VAE reaches SMAPE 143-181 (near the theoretical maximum of 200), making it essentially a random-output model. TrendVAE2+WaveletV3VAE2 is 59-113. The very short Tourism horizon (H=4) gives the model almost no room for error -- any noise in the backcast subtraction is devastating when there are only 4 forecast points.

**Weather:** Double-VAE is much less catastrophic (SMAPE 41-46 vs 38-43 for deterministic). On the longer Weather horizon (H=96), the reparameterization noise averages out more across the many forecast steps. The gap narrows from ~3-15x on M4/Tourism to ~1.1x on Weather.

In [ ]:
# Depth scaling across all 4 datasets (no-skip baselines only)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

depth_datasets = [
    ('Tourism v1 (TW alternating)', tourism_v1, 'TrendAELG_WaveletV3AELG', 'smape'),
    ('Tourism v2 (unified TWA/TWALG)', tourism_v2, None, 'smape'),
    ('Weather v1 (TW alternating)', weather_v1, 'TrendAELG_WaveletV3AELG', 'smape'),
    ('Weather v2 (unified TWA/TWALG)', weather_v2, None, 'smape'),
]

for ax, (title, dframe, arch_filter, metric) in zip(axes.flat, depth_datasets):
    best = get_best_round(dframe)
    noskip = best[~best['has_skip']]
    
    if arch_filter:
        archs = [arch_filter]
    else:
        archs = [a for a in ['TrendWaveletAE', 'TrendWaveletAELG', 'GenericAE'] 
                 if a in noskip['architecture'].unique()]
    
    for arch in archs:
        sub = noskip[noskip['architecture'] == arch]
        if len(sub) == 0:
            continue
        agg = sub.groupby('n_stacks')[metric].agg(['mean', 'std']).reset_index()
        color = ARCH_COLORS_EXT.get(arch, '#999')
        ax.errorbar(agg['n_stacks'], agg['mean'], yerr=agg['std'],
                    marker='o', label=ARCH_SHORT_EXT.get(arch, arch), color=color,
                    linewidth=2, capsize=4, markersize=8)
    
    ax.set_xlabel('Number of Stacks')
    ax.set_ylabel(metric.upper())
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Depth Scaling: No-Skip Baselines Across Datasets', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Summary table
print("\nOptimal depth by dataset (no-skip baselines):")
print("-" * 70)
for label, dframe, metric in [('Tourism v1', tourism_v1, 'smape'), ('Tourism v2', tourism_v2, 'smape'),
                                ('Weather v1', weather_v1, 'smape'), ('Weather v2', weather_v2, 'smape')]:
    best = get_best_round(dframe)
    noskip = best[~best['has_skip']]
    # Exclude extreme failures
    tw_archs = ['TrendAELG_WaveletV3AELG', 'TrendWaveletAE', 'TrendWaveletAELG']
    tw = noskip[noskip['architecture'].isin(tw_archs)]
    if len(tw) > 0:
        grp = tw.groupby('n_stacks')[metric].mean()
        best_depth = grp.idxmin()
        best_val = grp.min()
        worst_depth = grp.idxmax()
        worst_val = grp.max()
        print(f"  {label:15s} Best: {best_depth} stacks ({metric}={best_val:.3f}), "
              f"Worst: {worst_depth} stacks ({metric}={worst_val:.3f}), "
              f"Spread: {worst_val-best_val:.3f}")

## 16. Depth Scaling Across Datasets

One of the most important cross-dataset questions: does optimal stack depth depend on the forecast horizon?

- M4-Yearly (H=6): 10-30 stacks equivalent
- Tourism-Yearly (H=4): shallow (10-16) wins, depth hurts
- Weather-96 (H=96): moderate depth (16-20) optimal

If this pattern holds, it suggests a horizon-to-depth relationship: longer forecasts need more stacks to capture complex temporal patterns, while short forecasts can be modeled by fewer stacks and more depth just adds noise.

### Weather Skip Findings

**Weather-96 shows the most consistent no-skip advantage across all datasets:**

1. **No architecture significantly benefits from skip on Weather.** In v1, all three architecture families (TrendWav, GenericAELG, Generic) show no significant skip effect (all p > 0.4). In v2, skip actively hurts TrendVAE2+WaveletV3VAE2 (p=0.013) and TrendVAE+Haar (p=0.034).

2. **GenericAELG does NOT collapse on Weather** at 20-30 stacks (SMAPE 44-47 vs 40-44 for TrendWav). The bimodal failure mode seen on M4 does not reproduce here. Weather's longer sequences and normalization may stabilize the learned gate's gradient flow.

3. **GAELG10 has catastrophic MSE on Weather** (MSE ~2700 vs ~0.1 for TrendWav) despite having non-catastrophic SMAPE (~66). At 10 stacks, GenericAELG lacks the capacity to model Weather's 96-step forecast. The MSE explosion while SMAPE stays moderate suggests the model produces forecasts with roughly correct shape but wildly wrong scale.

4. **SMAPE and MSE rankings diverge on Weather v2.** TVH10 wins SMAPE (37.96) but TWALG20_skip3 wins MSE (0.09). This is because SMAPE is scale-invariant (it penalizes relative errors) while MSE penalizes absolute errors. For Weather (normalized data), MSE is arguably more meaningful.

5. **TrendWavelet at 16-20 stacks is optimal for Weather.** Both v1 (TW16_no_skip) and v2 (TWA20_no_skip, TWALG20_skip3) place TrendWavelet at 16-20 stacks in the top tier. Unlike M4 where 10-30 stacks were equivalent, Weather benefits from moderate depth.

In [ ]:
# Weather-96: Combined v1 + v2 final rankings (by both SMAPE and MSE)
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for col, metric, metric_label in [(0, 'smape', 'SMAPE'), (1, 'mse', 'MSE')]:
    for row, (label, dframe) in enumerate([('Weather v1', weather_v1), ('Weather v2', weather_v2)]):
        ax = axes[row, col]
        best = get_best_round(dframe)
        
        # Filter out extreme outliers for readability (MSE > 100)
        if metric == 'mse':
            best = best[best['mse'] < 100]
        
        summary = best.groupby(['config_name', 'architecture']).agg(
            mean_val=(metric, 'mean'),
            std_val=(metric, 'std'),
            n_runs=(metric, 'count'),
        ).sort_values('mean_val').reset_index()
        
        show = summary.head(18)
        colors = [ARCH_COLORS_EXT.get(a, '#999') for a in show['architecture']]
        
        bars = ax.barh(range(len(show)), show['mean_val'],
                       xerr=show['std_val'], capsize=2, color=colors,
                       edgecolor='white', linewidth=0.5)
        ax.set_yticks(range(len(show)))
        ax.set_yticklabels([f"{c} (n={n})" for c, n in zip(show['config_name'], show['n_runs'])], fontsize=7)
        ax.set_xlabel(metric_label)
        ax.set_title(f'{label} by {metric_label}')
        ax.invert_yaxis()
        
        for i, (s, std) in enumerate(zip(show['mean_val'], show['std_val'])):
            fmt = f'{s:.3f}' if metric == 'smape' else f'{s:.4f}'
            ax.text(s + std + (0.2 if metric == 'smape' else 0.002), i, fmt, va='center', fontsize=7)

patches = [mpatches.Patch(color=ARCH_COLORS_EXT.get(a, '#999'), label=ARCH_SHORT_EXT.get(a, a))
           for a in sorted(set(weather_v1['architecture'].unique()) | set(weather_v2['architecture'].unique()))
           if a in ARCH_COLORS_EXT]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Weather-96: Skip Study Rankings', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Print key weather results
for label, dframe in [('WEATHER V1', weather_v1), ('WEATHER V2', weather_v2)]:
    best = get_best_round(dframe)
    best_valid = best[best['mse'] < 100]  # exclude diverged
    summary = best_valid.groupby(['config_name', 'architecture', 'n_stacks']).agg(
        mean_smape=('smape', 'mean'), mean_mse=('mse', 'mean'), n=('smape', 'count')
    ).sort_values('mean_smape').reset_index()
    print(f"\n{label} Top 10 by SMAPE:")
    for _, row in summary.head(10).iterrows():
        skip = 'skip' if 'skip' in row['config_name'] and 'no_skip' not in row['config_name'] else 'no_skip'
        print(f"  {row['config_name']:30s} SMAPE={row['mean_smape']:.3f} MSE={row['mean_mse']:.4f} "
              f"[{ARCH_SHORT_EXT.get(row['architecture'],'?')}, {row['n_stacks']}st, {skip}]")

## 15. Weather-96: Final Rankings

Weather-96 provides the opposite extreme from Tourism: H=96, L=480, with 21 multivariate meteorological variables. Deep stacks might matter more here due to the complex, long-horizon signal.

**v1 winner:** TW16_no_skip (SMAPE 40.25) -- TrendAELG+WaveletV3AELG alternating, no skip.

**v2 winner (SMAPE):** TVH10_no_skip (SMAPE 37.96) -- surprisingly, TrendVAE+HaarWaveletV3 at only 10 stacks.

**v2 winner (MSE):** TWALG20_skip3_a01 (MSE 0.09) -- TrendWaveletAELG with skip, diverging from the SMAPE ranking.

Note: Weather v2 R3 is incomplete (only 2 of 9 expected configs reached R3). Most v2 results are from R2 (30 epochs, 3 runs), limiting statistical power.

### Tourism Skip Findings

**Tourism is the first dataset where the M4 conclusions partially diverge:**

1. **TrendWaveletAE: no-skip is significantly better** (MWU p=0.001, r=0.79). Skip actively hurts on Tourism. This matches M4.

2. **GenericAE: no-skip is significantly better** (MWU p=0.011, r=0.55). Same as M4.

3. **Generic legacy: skip DOES help on Tourism** (p=0.014). On M4, skip had negligible effect on Generic. On Tourism's very short horizon (H=4), the Generic block without inductive bias benefits from the residual re-injection, perhaps because the signal is so compact that deep stacks lose it without skip.

4. **Double-VAE: skip helps on Tourism** (TrendVAE+WaveletV3VAE p=0.010, TrendVAE2+WaveletV3VAE2 p=0.003). However, even with skip, these architectures achieve SMAPE 46-150 -- still catastrophically bad. Skip reduces the severity of double-VAE failure but does not fix it.

5. **GenericAELG: skip marginally helps** (ns, p=0.49). On M4, skip was critical for GAELG at depth>=20. On Tourism, the bimodal collapse is less severe (only 1/3 seeds fail at 30 stacks vs 2/3 on M4), and the shorter horizon may prevent full catastrophe.

**Key insight:** On short-horizon Tourism, **shallow architectures win**. GAE10_no_skip (10 stacks, SMAPE 20.53) beats everything else including all skip configs. The optimal depth is 10-16 stacks, much less than M4's 16-30.

In [ ]:
# Skip vs No-Skip statistical tests for Tourism and Weather
from scipy.stats import mannwhitneyu

def skip_vs_noskip_analysis(dframe, metric, dataset_label):
    """Run MWU tests for skip vs no-skip per architecture."""
    best = get_best_round(dframe)
    results = []
    
    for arch in sorted(best['architecture'].unique()):
        sub = best[best['architecture'] == arch]
        noskip = sub[~sub['has_skip']][metric].values
        skip = sub[sub['has_skip']][metric].values
        
        if len(noskip) < 3 or len(skip) < 3:
            continue
        
        u_stat, p_val = mannwhitneyu(noskip, skip, alternative='two-sided')
        n1, n2 = len(noskip), len(skip)
        r_rb = 1 - (2 * u_stat) / (n1 * n2)  # rank-biserial
        
        results.append({
            'Architecture': ARCH_SHORT_EXT.get(arch, arch),
            'No-Skip Mean': np.mean(noskip),
            'Skip Mean': np.mean(skip),
            'Delta': np.mean(skip) - np.mean(noskip),
            'n_noskip': n1,
            'n_skip': n2,
            'MWU p': p_val,
            'rank-biserial r': r_rb,
            'Winner': 'NO-SKIP' if np.mean(noskip) < np.mean(skip) else 'SKIP',
            'Sig': '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns',
        })
    
    res_df = pd.DataFrame(results)
    print(f"\n{'='*90}")
    print(f"{dataset_label}: Skip vs No-Skip ({metric.upper()})")
    print(f"{'='*90}")
    if len(res_df) > 0:
        print(res_df.to_string(index=False, float_format='%.4f'))
    else:
        print("  No testable architecture groups found.")
    return res_df

# Tourism
t1_tests = skip_vs_noskip_analysis(tourism_v1, 'smape', 'Tourism v1')
t2_tests = skip_vs_noskip_analysis(tourism_v2, 'smape', 'Tourism v2')

# Weather
w1_tests = skip_vs_noskip_analysis(weather_v1, 'smape', 'Weather v1')
w2_tests = skip_vs_noskip_analysis(weather_v2, 'smape', 'Weather v2')

# Visualization: Delta SMAPE (skip - noskip) by architecture and dataset
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, (label, tests_df) in zip(axes.flat, 
    [('Tourism v1', t1_tests), ('Tourism v2', t2_tests),
     ('Weather v1', w1_tests), ('Weather v2', w2_tests)]):
    
    if len(tests_df) == 0:
        ax.set_title(f'{label}: No testable groups')
        continue
    
    colors = ['#4CAF50' if d < 0 else '#F44336' for d in tests_df['Delta']]
    bars = ax.barh(range(len(tests_df)), tests_df['Delta'], color=colors, alpha=0.8, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(range(len(tests_df)))
    ax.set_yticklabels([f"{a} ({s})" for a, s in zip(tests_df['Architecture'], tests_df['Sig'])], fontsize=9)
    ax.set_xlabel('Delta SMAPE (Skip - NoSkip)')
    ax.set_title(f'{label}: Skip Effect by Architecture')
    
    for i, (d, sig) in enumerate(zip(tests_df['Delta'], tests_df['Sig'])):
        ax.text(d + (0.3 if d > 0 else -0.3), i, f'{d:+.2f}', va='center', fontsize=8,
                ha='left' if d > 0 else 'right')

plt.suptitle('Skip Connection Effect: Green = Skip Helps, Red = Skip Hurts', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 14. Tourism: Skip vs No-Skip Statistical Tests

For each architecture family on Tourism, we test whether skip connections significantly change SMAPE. We use Mann-Whitney U (two-sided) comparing all skip runs against all no-skip runs at their best achieved round.

**Hypothesis:** Based on M4, we expect skip to be unnecessary for TrendWavelet and GenericAE, but potentially helpful for GenericAELG (which showed depth-collapse on M4).

In [ ]:
# Tourism-Yearly: Combined v1 + v2 final rankings
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

for ax, (label, dframe) in zip(axes, [('Tourism v1 (best round per config)', tourism_v1),
                                        ('Tourism v2 (best round per config)', tourism_v2)]):
    best = get_best_round(dframe)
    summary = best.groupby(['config_name', 'architecture']).agg(
        mean_smape=('smape', 'mean'),
        std_smape=('smape', 'std'),
        n_runs=('smape', 'count'),
    ).sort_values('mean_smape').reset_index()
    
    # Truncate to top 20 for readability
    show = summary.head(20)
    colors = [ARCH_COLORS_EXT.get(a, '#999') for a in show['architecture']]
    
    bars = ax.barh(range(len(show)), show['mean_smape'], 
                   xerr=show['std_smape'], capsize=3, color=colors, 
                   edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(show)))
    ax.set_yticklabels([f"{c} (n={n})" for c, n in zip(show['config_name'], show['n_runs'])], fontsize=8)
    ax.set_xlabel('SMAPE')
    ax.set_title(label)
    ax.invert_yaxis()
    
    for i, (s, std) in enumerate(zip(show['mean_smape'], show['std_smape'])):
        ax.text(s + std + 0.05, i, f'{s:.3f}', va='center', fontsize=8)

# Add SOTA reference line to v2 plot
axes[1].axvline(20.864, color='red', linestyle='--', alpha=0.6, linewidth=1.5, label='SOTA (20.864)')
axes[1].legend(fontsize=9)

# Legend for both plots
patches = [mpatches.Patch(color=ARCH_COLORS_EXT.get(a, '#999'), label=ARCH_SHORT_EXT.get(a, a)) 
           for a in sorted(set(tourism_v1['architecture'].unique()) | set(tourism_v2['architecture'].unique()))
           if a in ARCH_COLORS_EXT]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Tourism-Yearly: Skip Study Rankings', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Print complete rankings
for label, dframe in [('TOURISM V1', tourism_v1), ('TOURISM V2', tourism_v2)]:
    best = get_best_round(dframe)
    summary = best.groupby(['config_name', 'architecture', 'n_stacks']).agg(
        mean_smape=('smape', 'mean'), std_smape=('smape', 'std'), n=('smape', 'count')
    ).sort_values('mean_smape').reset_index()
    print(f"\n{label} Top 10:")
    for i, row in summary.head(10).iterrows():
        skip = 'skip' if 'skip' in row['config_name'] and 'no_skip' not in row['config_name'] else 'no_skip'
        delta = f"+{row['mean_smape'] - summary['mean_smape'].iloc[0]:.3f}" if i > 0 else "BEST"
        print(f"  {row['config_name']:30s} SMAPE={row['mean_smape']:.3f} +/-{row['std_smape']:.3f} "
              f"(n={row['n']}) [{ARCH_SHORT_EXT.get(row['architecture'],'?')}, "
              f"{row['n_stacks']}st, {skip}] {delta}")

## 13. Tourism-Yearly: Final Rankings

Tourism-Yearly is a particularly interesting test case: with H=4 and L=8, it has the shortest horizon and lookback in the study. Deep stacks (20-30) may be excessive for such a short signal, potentially reversing the depth-scaling conclusions from M4.

**v1 winner:** GAELG20_skip5_learn (SMAPE 21.17) -- notably, a GenericAELG config WITH skip, unlike M4 where TrendWavelet dominated.

**v2 winner:** GAE10_no_skip (SMAPE 20.53) -- GenericAE at the shallowest depth, without skip.

**Tourism-Yearly SOTA (from other studies):** TrendWaveletAELG_coif3_eq_bcast_td3_ld16, SMAPE=20.864. The GAE10 result here at 20.53 is potentially competitive.

In [ ]:
# ==================================================================================
# Load Tourism and Weather data (v1 and v2 for both datasets)
# ==================================================================================
import os
BASE = '/home/realdanielbyrne/GitHub/N-BEATS-Lightning/experiments/results'

def load_combined(ds, ver):
    """Load primary dataset-named file, supplement unique rows from generic-named file."""
    p1 = os.path.join(BASE, ds, f'resnet_skip_study{ver}_{ds}_results.csv')
    p2 = os.path.join(BASE, ds, f'resnet_skip_study{ver}_results.csv')
    d1 = pd.read_csv(p1)
    d2 = pd.read_csv(p2)
    shared = list(set(d1.columns) & set(d2.columns))
    keys = ['config_name', 'run', 'seed', 'search_round']
    existing_keys = [k for k in keys if k in d1.columns and k in d2.columns]
    return pd.concat([d1[shared], d2[shared]], ignore_index=True).drop_duplicates(
        subset=existing_keys, keep='first')

tourism_v1 = load_combined('tourism', '')
tourism_v2 = load_combined('tourism', '_v2')
weather_v1 = load_combined('weather', '')
weather_v2 = load_combined('weather', '_v2')

# Ensure consistent dtypes
for df_name, dframe in [('tourism_v1', tourism_v1), ('tourism_v2', tourism_v2),
                          ('weather_v1', weather_v1), ('weather_v2', weather_v2)]:
    dframe['n_stacks'] = dframe['n_stacks'].astype(int)
    dframe['skip_distance_cfg'] = pd.to_numeric(dframe['skip_distance_cfg'], errors='coerce').fillna(0).astype(int)
    dframe['search_round'] = dframe['search_round'].astype(int)
    dframe['has_skip'] = dframe['skip_distance_cfg'] > 0

def get_best_round(dframe):
    """Get data at each config's best (latest) round."""
    max_r = dframe.groupby('config_name')['search_round'].max().reset_index()
    max_r.columns = ['config_name', 'mr']
    merged = dframe.merge(max_r, on='config_name')
    return merged[merged['search_round'] == merged['mr']].drop(columns=['mr'])

# Extended color palette for v1 architectures
ARCH_COLORS_EXT = {
    **ARCH_COLORS,
    'TrendAELG_WaveletV3AELG': '#1565C0',  # Dark blue
    'GenericAELG': '#E91E63',                # Pink  
    'Generic': '#607D8B',                    # Blue-grey
}

ARCH_SHORT_EXT = {
    **ARCH_SHORT,
    'TrendAELG_WaveletV3AELG': 'TW-alt',
    'GenericAELG': 'GAELG',
    'Generic': 'G30',
}

print("Loaded datasets:")
for name, df in [('Tourism v1', tourism_v1), ('Tourism v2', tourism_v2),
                  ('Weather v1', weather_v1), ('Weather v2', weather_v2)]:
    rounds = df.groupby('search_round')['config_name'].nunique()
    print(f"  {name}: {len(df)} rows, {df['config_name'].nunique()} configs | "
          f"Rounds: {dict(rounds)}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Consistent styling
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Architecture color palette (consistent across all plots)
ARCH_COLORS = {
    'TrendWaveletAELG': '#2196F3',       # Blue
    'TrendWaveletAE': '#4CAF50',          # Green
    'GenericAE': '#FF9800',               # Orange
    'TrendVAE_HaarWaveletV3': '#9C27B0',  # Purple
    'TrendVAE_WaveletV3VAE': '#F44336',   # Red
    'TrendVAE2_WaveletV3VAE2': '#795548', # Brown
}

ARCH_SHORT = {
    'TrendWaveletAELG': 'TWALG',
    'TrendWaveletAE': 'TWA',
    'GenericAE': 'GAE',
    'TrendVAE_HaarWaveletV3': 'TVH',
    'TrendVAE_WaveletV3VAE': 'TVAE',
    'TrendVAE2_WaveletV3VAE2': 'TV2',
}

# Load data
CSV_PATH = '/home/realdanielbyrne/GitHub/N-BEATS-Lightning/experiments/results/m4/resnet_skip_study_v2_results.csv'
df = pd.read_csv(CSV_PATH)

# Parse n_stacks as int
df['n_stacks'] = df['n_stacks'].astype(int)
df['skip_distance_cfg'] = df['skip_distance_cfg'].astype(int)
df['search_round'] = df['search_round'].astype(int)

# Add derived columns
df['has_skip'] = df['skip_distance_cfg'] > 0
df['arch_short'] = df['architecture'].map(ARCH_SHORT)

# Split by round
r1 = df[df['search_round'] == 1].copy()
r2 = df[df['search_round'] == 2].copy()
r3 = df[df['search_round'] == 3].copy()

print(f"Total runs: {len(df)}")
print(f"Round 1: {r1['config_name'].nunique()} configs, {len(r1)} runs (15 epochs)")
print(f"Round 2: {r2['config_name'].nunique()} configs, {len(r2)} runs (30 epochs)")
print(f"Round 3: {r3['config_name'].nunique()} configs, {len(r3)} runs (100 epochs, early stop)")
print(f"\nArchitectures: {df['architecture'].nunique()}")
print(f"Diverged runs: {df['diverged'].sum()}")

# ResNet Skip Connection Study v2 -- Comprehensive Analysis

**Dataset:** M4-Yearly (H=6, L=30)  
**Date:** 2026-03-07  
**Method:** 3-round successive halving (36 configs: 15/30/100 epochs)  
**Results:** `experiments/results/m4/resnet_skip_study_v2_results.csv`  
**Config:** `experiments/configs/resnet_skip_study_v2.yaml`

## Study Motivation

The v1 skip connection study (24 configs) established that skip connections rescue **GenericAELG** at depth >= 20 stacks (SMAPE 36 -> 13.8) but do not help TrendAELG+WaveletV3AELG or Generic, which are inherently depth-stable. This v2 follow-up fills gaps identified in v1:

| Section | Architecture | Configs | Question |
|---------|-------------|---------|----------|
| A | GenericAE (homogeneous) | 6 | Does GenericAE degrade at depth like GenericAELG? |
| B | TrendVAE + WaveletV3VAE | 6 | Does double-VAE interact with skip? |
| C | TrendVAE2 + WaveletV3VAE2 | 6 | Does double-VAE2 (known unstable) benefit from skip? |
| D | TrendWaveletAE (homogeneous) | 6 | Combined block depth scaling + short skip distances |
| E | TrendWaveletAELG (homogeneous) | 6 | LG variant depth scaling -- gate effect vs. non-LG |
| F | TrendVAE + HaarWaveletV3 | 6 | Corrected single-VAE design (VAE + deterministic wavelet) |

## 1. All Results Overview

All 36 configs ranked by mean SMAPE across all rounds. The `max_round` column shows how far each config progressed through successive halving (3 = finalist). CV% (coefficient of variation) measures run-to-run reproducibility -- lower is better.

In [ ]:
# Summary table: all configs, aggregated across all rounds
summary_all = df.groupby(['config_name', 'architecture', 'n_stacks', 'skip_distance_cfg']).agg(
    mean_smape=('smape', 'mean'),
    std_smape=('smape', 'std'),
    mean_owa=('owa', 'mean'),
    n_runs=('smape', 'count'),
    max_round=('search_round', 'max'),
    mean_params=('n_params', 'first'),
).sort_values('mean_smape').reset_index()

summary_all['cv_pct'] = (summary_all['std_smape'] / summary_all['mean_smape'] * 100).round(1)
summary_all['skip'] = summary_all['skip_distance_cfg'].apply(lambda x: f'd={x}' if x > 0 else '--')

display_cols = ['config_name', 'architecture', 'n_stacks', 'skip', 'mean_smape', 'std_smape', 'cv_pct', 'mean_owa', 'n_runs', 'max_round', 'mean_params']
print("All 36 Configs Ranked by Mean SMAPE (all rounds pooled):")
print("=" * 120)
print(summary_all[display_cols].to_string(index=False, float_format='%.3f'))

## 2. R3 Final Rankings (Definitive Results)

Only 9 of 36 configs survived to Round 3 (100 epochs, 5 runs each). All 9 finalists are TrendWavelet family blocks -- every other architecture was eliminated by R2. The top 3 are all **no-skip baselines**, indicating that skip connections provide no benefit for depth-stable architectures.

Note the tight SMAPE spread: only 0.23 points separate #1 from #9, suggesting the TrendWavelet family has a consistent performance ceiling around SMAPE ~13.6 regardless of depth (10-30 stacks) or skip configuration.

In [ ]:
# R3 final rankings bar chart
r3_summary = r3.groupby(['config_name', 'architecture']).agg(
    mean_smape=('smape', 'mean'),
    std_smape=('smape', 'std'),
    mean_owa=('owa', 'mean'),
).sort_values('mean_smape').reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
colors = [ARCH_COLORS[a] for a in r3_summary['architecture']]
bars = ax.barh(range(len(r3_summary)), r3_summary['mean_smape'], 
               xerr=r3_summary['std_smape'], capsize=3, color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(r3_summary)))
ax.set_yticklabels(r3_summary['config_name'], fontsize=10)
ax.set_xlabel('SMAPE')
ax.set_title('Round 3 Final Rankings (100 epochs, 5 runs each)')
ax.invert_yaxis()

# Add value labels
for i, (s, std) in enumerate(zip(r3_summary['mean_smape'], r3_summary['std_smape'])):
    ax.text(s + std + 0.02, i, f'{s:.3f}', va='center', fontsize=9)

# Legend
patches = [mpatches.Patch(color=c, label=ARCH_SHORT[a]) for a, c in ARCH_COLORS.items() 
           if a in r3_summary['architecture'].values]
ax.legend(handles=patches, loc='lower right', framealpha=0.9)
plt.tight_layout()
plt.show()

## 3. Successive Halving Progression

Three rounds of elimination (keep 50% each round): 36 -> 18 -> 9 configs.

**R1 -> R2 (eliminated 18):** All double-VAE (TVAE, TV2) and double-VAE2 configs were eliminated immediately -- their SMAPE 29-44 made them obvious cuts. GenericAE baselines at 10/20 stacks, and most TVH (TrendVAE+Haar) skip variants also fell.

**R2 -> R3 (eliminated 9):** The remaining GenericAE configs (including GAE30_skip3, the best GAE variant), the last TVH configs, TWA10_no_skip, and all skip_distance=2 variants were cut. By R3, only TrendWaveletAE and TrendWaveletAELG remained.

Hatched bars indicate configs eliminated in that round.

In [ ]:
# Successive halving Sankey-style visualization
r1_configs = set(r1['config_name'].unique())
r2_configs = set(r2['config_name'].unique())
r3_configs = set(r3['config_name'].unique())

# Get mean SMAPE per config per round for ordering
def round_summary(rdf):
    return rdf.groupby(['config_name', 'architecture'])['smape'].mean().reset_index().sort_values('smape')

r1_sum = round_summary(r1)
r2_sum = round_summary(r2)
r3_sum = round_summary(r3)

fig, axes = plt.subplots(1, 3, figsize=(18, 10), sharey=False)

for ax, rsum, title, n_total in zip(axes, [r1_sum, r2_sum, r3_sum],
    ['Round 1 (15 ep, 36 configs)', 'Round 2 (30 ep, 18 configs)', 'Round 3 (100 ep, 9 configs)'],
    [36, 18, 9]):
    
    colors = [ARCH_COLORS.get(a, '#999') for a in rsum['architecture']]
    survived_r2 = [c in r2_configs for c in rsum['config_name']]
    survived_r3 = [c in r3_configs for c in rsum['config_name']]
    
    bars = ax.barh(range(len(rsum)), rsum['smape'], color=colors, edgecolor='white', linewidth=0.5)
    
    # Mark eliminated configs with hatching
    for i, (bar, cfg) in enumerate(zip(bars, rsum['config_name'])):
        if rsum is r1_sum and cfg not in r2_configs:
            bar.set_hatch('///')
            bar.set_alpha(0.4)
        elif rsum is r2_sum and cfg not in r3_configs:
            bar.set_hatch('///')
            bar.set_alpha(0.4)
    
    ax.set_yticks(range(len(rsum)))
    ax.set_yticklabels(rsum['config_name'], fontsize=7)
    ax.set_xlabel('Mean SMAPE')
    ax.set_title(title, fontsize=11)
    ax.invert_yaxis()

# Legend
patches = [mpatches.Patch(color=c, label=f'{ARCH_SHORT[a]} ({a})') for a, c in ARCH_COLORS.items()]
patches.append(mpatches.Patch(facecolor='gray', hatch='///', alpha=0.4, label='Eliminated'))
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Successive Halving Progression', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Print elimination summary
print("\nEliminated after R1 (18):")
for c in sorted(r1_configs - r2_configs):
    arch = r1[r1['config_name']==c]['architecture'].iloc[0]
    smape = r1[r1['config_name']==c]['smape'].mean()
    print(f"  {c:25s} ({ARCH_SHORT[arch]})  SMAPE={smape:.2f}")

print(f"\nEliminated after R2 (9):")
for c in sorted(r2_configs - r3_configs):
    arch = r2[r2['config_name']==c]['architecture'].iloc[0]
    smape = r2[r2['config_name']==c]['smape'].mean()
    print(f"  {c:25s} ({ARCH_SHORT[arch]})  SMAPE={smape:.2f}")

## 4. Per-Architecture Box Plots

The R1 data (all 36 configs, 15 epochs, 3 runs each) gives the broadest view of architecture-level performance. The box plot reveals a clear **three-tier structure**:

- **Tier 1 (SMAPE ~14):** TrendWaveletAE and TrendWaveletAELG -- tight distributions, low variance
- **Tier 2 (SMAPE ~15):** GenericAE and TrendVAE+HaarWaveletV3 -- viable but clearly worse
- **Tier 3 (SMAPE 29-44):** Double-VAE and Double-VAE2 -- catastrophic failures, 2-3x worse than Tier 1

The gap between Tier 2 and Tier 3 is the most striking feature: ~15 SMAPE points separate the single-VAE design (TVH) from the double-VAE design (TVAE), confirming that the problem is specifically the **pairing of two VAE-backbone blocks**, not the VAE backbone itself.

In [ ]:
# Box plots by architecture (R1 only -- all 36 configs present)
arch_order = r1.groupby('architecture')['smape'].mean().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(14, 7))
bp_data = [r1[r1['architecture'] == a]['smape'].values for a in arch_order]
bp = ax.boxplot(bp_data, labels=[ARCH_SHORT[a] for a in arch_order], 
                patch_artist=True, widths=0.6, showfliers=True,
                flierprops=dict(marker='o', markersize=4, alpha=0.5))

for patch, arch in zip(bp['boxes'], arch_order):
    patch.set_facecolor(ARCH_COLORS[arch])
    patch.set_alpha(0.7)

# Overlay individual points
for i, (arch, data) in enumerate(zip(arch_order, bp_data)):
    jitter = np.random.normal(0, 0.05, len(data))
    ax.scatter(np.full_like(data, i+1) + jitter, data, 
               color=ARCH_COLORS[arch], alpha=0.4, s=20, zorder=3)

ax.set_ylabel('SMAPE')
ax.set_title('SMAPE Distribution by Architecture (Round 1, all configs)')
ax.set_xlabel('Architecture')

# Add median labels
for i, data in enumerate(bp_data):
    med = np.median(data)
    ax.text(i+1, med - 0.5, f'{med:.1f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Skip vs No-Skip Paired Comparison

For each architecture that has both skip and no-skip variants at the same depth, we compute the delta SMAPE (skip minus no-skip). Negative = skip helps, positive = skip hurts.

**What to look for:**
- v1 showed skip rescues GenericAELG at depth (delta = -22 SMAPE at 30 stacks). Does any v2 architecture show a similar rescue effect?
- Are short skip distances (d=2, d=3) better than the d=4-5 tested in v1?

**Key expectations:**
- GenericAE should show modest benefit (it doesn't collapse like GenericAELG, but skip might still help)
- TrendWavelet should show negligible or negative effect (depth-stable architecture)
- Double-VAE should show no benefit (fundamental design issue, not depth-related)

In [ ]:
# Skip vs no-skip paired comparison
# Use highest-round data available for each config
def get_best_round(cfg_name):
    sub = df[df['config_name'] == cfg_name]
    max_round = sub['search_round'].max()
    return sub[sub['search_round'] == max_round]

# Define pairs: (no_skip_config, skip_config, architecture, stacks, skip_dist)
pairs = [
    # GenericAE
    ('GAE30_no_skip', 'GAE30_skip3_a01', 'GenericAE', 30, 3),
    ('GAE20_no_skip', 'GAE20_skip5_a01', 'GenericAE', 20, 5),
    ('GAE30_no_skip', 'GAE30_skip5_a01', 'GenericAE', 30, 5),
    # TrendWaveletAE
    ('TWA20_no_skip', 'TWA20_skip3_a01', 'TrendWaveletAE', 20, 3),
    ('TWA30_no_skip', 'TWA30_skip3_a01', 'TrendWaveletAE', 30, 3),
    ('TWA30_no_skip', 'TWA30_skip2_a01', 'TrendWaveletAE', 30, 2),
    # TrendWaveletAELG
    ('TWALG20_no_skip', 'TWALG20_skip3_a01', 'TrendWaveletAELG', 20, 3),
    ('TWALG30_no_skip', 'TWALG30_skip3_a01', 'TrendWaveletAELG', 30, 3),
    ('TWALG30_no_skip', 'TWALG30_skip2_a01', 'TrendWaveletAELG', 30, 2),
    # TrendVAE+WaveletV3VAE
    ('TVAE16_no_skip', 'TVAE16_skip4_a01', 'TrendVAE_WaveletV3VAE', 16, 4),
    ('TVAE24_no_skip', 'TVAE24_skip4_a01', 'TrendVAE_WaveletV3VAE', 24, 4),
    ('TVAE24_no_skip', 'TVAE24_skip3_a01', 'TrendVAE_WaveletV3VAE', 24, 3),
    # TrendVAE+HaarWaveletV3
    ('TVH20_no_skip', 'TVH20_skip5_a01', 'TrendVAE_HaarWaveletV3', 20, 5),
    ('TVH30_no_skip', 'TVH30_skip5_a01', 'TrendVAE_HaarWaveletV3', 30, 5),
    ('TVH30_no_skip', 'TVH30_skip3_a01', 'TrendVAE_HaarWaveletV3', 30, 3),
]

fig, ax = plt.subplots(figsize=(14, 8))
y_labels = []
deltas = []
colors_list = []

for i, (base_cfg, skip_cfg, arch, stacks, sdist) in enumerate(pairs):
    base_data = r1[r1['config_name'] == base_cfg]['smape']
    skip_data = r1[r1['config_name'] == skip_cfg]['smape']
    
    if len(base_data) == 0 or len(skip_data) == 0:
        continue
    
    delta = skip_data.mean() - base_data.mean()
    deltas.append(delta)
    y_labels.append(f'{ARCH_SHORT[arch]}{stacks} d={sdist}')
    colors_list.append(ARCH_COLORS[arch])

bars = ax.barh(range(len(deltas)), deltas, color=colors_list, edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(range(len(deltas)))
ax.set_yticklabels(y_labels, fontsize=10)
ax.set_xlabel('Delta SMAPE (skip - no_skip)')
ax.set_title('Effect of Skip Connections: Negative = Skip Helps')

for i, d in enumerate(deltas):
    side = 'left' if d > 0 else 'right'
    offset = 0.1 if d > 0 else -0.1
    ax.text(d + offset, i, f'{d:+.2f}', va='center', ha=side, fontsize=9)

plt.tight_layout()
plt.show()

### Interpretation

**No architecture shows the dramatic rescue effect seen in v1 for GenericAELG (delta = -22).** The largest benefit is GAE30 d=3 at -0.40 SMAPE -- meaningful but not transformative. This confirms that GenericAE does not suffer the depth collapse that GenericAELG does.

**TrendWavelet results are mixed-to-negative.** TWALG20 d=3 shows -0.16 (marginal help), but TWALG30 d=3 shows +0.09 and TWA30 d=3 shows +0.17 (skip hurts). The integrated polynomial+DWT basis in TrendWavelet provides sufficient inductive bias to prevent residual decay, so adding skip noise degrades the signal.

**skip_distance=2 consistently hurts** (both TWA30 d=2 and TWALG30 d=2 are positive). This is too frequent -- the skip injection disrupts the residual stream before blocks have enough layers to process the information.

**Double-VAE skip effects are large but wrong-direction.** TVAE skip variants are +2 to +5 SMAPE worse than no-skip baselines. The problem is not depth-related signal decay (which skip addresses) but compounded reparameterization noise (which skip cannot fix).

## 6. Depth Scaling Analysis

A central question of this study: **how does SMAPE change as we scale from 10 to 30 stacks?**

The v1 study showed GenericAELG collapsing catastrophically at 20+ stacks (SMAPE 14 -> 36). Does this affect other architectures?

**Three panels:**
- **Left:** Deterministic architectures (TWA, TWALG, GAE) -- do they scale cleanly?
- **Center:** VAE architectures (TVAE, TV2, TVH) -- does depth compound the stochastic noise problem?
- **Right:** Best no-skip performance per architecture -- overall hierarchy at optimal depth

In [ ]:
# Depth scaling: SMAPE vs n_stacks for no-skip baselines
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Deterministic architectures (R1 baselines only)
ax = axes[0]
for arch in ['TrendWaveletAE', 'TrendWaveletAELG', 'GenericAE']:
    baseline = r1[(r1['architecture'] == arch) & (~r1['has_skip'])]
    if len(baseline) == 0:
        continue
    agg = baseline.groupby('n_stacks').agg(mean=('smape','mean'), std=('smape','std')).reset_index()
    ax.errorbar(agg['n_stacks'], agg['mean'], yerr=agg['std'], 
                marker='o', label=ARCH_SHORT[arch], color=ARCH_COLORS[arch],
                linewidth=2, capsize=4, markersize=8)

ax.set_xlabel('Number of Stacks')
ax.set_ylabel('SMAPE')
ax.set_title('Depth Scaling: Deterministic Architectures')
ax.legend()
ax.set_xticks([10, 20, 30])
ax.grid(alpha=0.3)

# Panel 2: VAE architectures
ax = axes[1]
for arch in ['TrendVAE_WaveletV3VAE', 'TrendVAE2_WaveletV3VAE2', 'TrendVAE_HaarWaveletV3']:
    baseline = r1[(r1['architecture'] == arch) & (~r1['has_skip'])]
    if len(baseline) == 0:
        continue
    agg = baseline.groupby('n_stacks').agg(mean=('smape','mean'), std=('smape','std')).reset_index()
    ax.errorbar(agg['n_stacks'], agg['mean'], yerr=agg['std'], 
                marker='s', label=ARCH_SHORT[arch], color=ARCH_COLORS[arch],
                linewidth=2, capsize=4, markersize=8)

ax.set_xlabel('Number of Stacks')
ax.set_ylabel('SMAPE')
ax.set_title('Depth Scaling: VAE Architectures')
ax.legend()
ax.grid(alpha=0.3)

# Panel 3: All architectures at 10 stacks (or lowest) -- normalized comparison
ax = axes[2]
arch_best = {}
for arch in ARCH_COLORS.keys():
    baseline = r1[(r1['architecture'] == arch) & (~r1['has_skip'])]
    if len(baseline) > 0:
        agg = baseline.groupby('n_stacks')['smape'].mean()
        arch_best[ARCH_SHORT[arch]] = agg.min()

sorted_archs = sorted(arch_best.items(), key=lambda x: x[1])
names = [a[0] for a in sorted_archs]
vals = [a[1] for a in sorted_archs]
full_names = {v: k for k, v in ARCH_SHORT.items()}
colors = [ARCH_COLORS[full_names[n]] for n in names]

ax.barh(range(len(names)), vals, color=colors, edgecolor='white')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Best Baseline SMAPE (any depth)')
ax.set_title('Best No-Skip Performance by Architecture')
ax.invert_yaxis()
for i, v in enumerate(vals):
    ax.text(v + 0.2, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### Interpretation

**Deterministic architectures (left panel):**
- **TrendWaveletAE and TrendWaveletAELG are remarkably flat** from 10 to 30 stacks. The integrated polynomial + DWT basis provides sufficient inductive bias to prevent residual decay at any depth. This is the key finding of the study.
- **GenericAE is also stable** -- no systematic degradation. SMAPE stays between 15.2-16.2 across all depths. This contrasts sharply with GenericAELG's collapse in v1 (14 -> 36 at 30 stacks). The learned gate in AERootBlockLG appears to amplify gradient issues at depth, while the plain AERootBlock is inherently stable.

**VAE architectures (center panel):**
- **Double-VAE (TVAE) starts bad and stays bad** across all depths (29-32). Depth doesn't make things worse because the problem is already at the block level, not the stack level.
- **Double-VAE2 (TV2) is even worse** (37-44) and shows no clear depth trend.
- **TVH (single-VAE)** is the only VAE variant that functions reasonably (15.3), but degrades and becomes unstable at depth (TVH30_skip3 = 16.4 +/- 1.4).

**Architecture hierarchy (right panel):**
The ranking is clear and consistent: TrendWaveletAE/AELG (~14.2) > GenericAE (~15.2) > TVH (~15.3) >> double-VAE (29+) >> double-VAE2 (37+). This matches the established backbone hierarchy: RootBlock > AELG >= AE >> VAE.

## 7. Double-VAE vs Single-VAE vs Deterministic

This is perhaps the most practically important finding of the study. The prior `lg_vae_study` successfully paired TrendVAE with **deterministic** wavelets (HaarWaveletV3, a RootBlock descendant), achieving SMAPE ~14.2. But when both blocks use VAE backbones, performance collapses.

**The mechanism:** N-BEATS relies on precise backcast subtraction: `residual = input - backcast`. Each block's backcast must faithfully reconstruct the portion of the signal it captured. With VAE backbones, the reparameterization trick (`z = mu + std * eps`) injects stochastic noise into the latent representation. A single VAE block's noise is manageable -- the deterministic wavelet block in the next stack can still work with the slightly-noisy residual. But when **both** blocks use VAE backbones, noise compounds: the wavelet VAE receives a noisy residual from the trend VAE, adds its own noise, and passes an even noisier residual downstream. Over 8-24 stacks, this corruption accumulates catastrophically.

**Left panel:** Bar chart comparing matched-depth configs at their smallest stack counts.  
**Right panel:** Parameter efficiency (SMAPE vs model size) -- does more capacity help?

In [ ]:
# VAE design comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Bar chart of matched-depth comparisons
ax = axes[0]
comparisons = {
    'Double-VAE\n(TVAE, 8 stacks)': ('TVAE8_no_skip', r1),
    'Double-VAE2\n(TV2, 8 stacks)': ('TV2_8_no_skip', r1),
    'Single-VAE\n(TVH, 10 stacks)': ('TVH10_no_skip', r1),
    'TrendWaveletAE\n(TWA, 10 stacks)': ('TWA10_no_skip', r1),
    'TrendWaveletAELG\n(TWALG, 10 stacks)': ('TWALG10_no_skip', r1),
}

names = list(comparisons.keys())
means = []
stds = []
bar_colors = []
arch_map = {'TVAE': 'TrendVAE_WaveletV3VAE', 'TV2': 'TrendVAE2_WaveletV3VAE2',
            'TVH': 'TrendVAE_HaarWaveletV3', 'TWA': 'TrendWaveletAE', 'TWALG': 'TrendWaveletAELG'}

for label, (cfg, rdf) in comparisons.items():
    sub = rdf[rdf['config_name'] == cfg]
    means.append(sub['smape'].mean())
    stds.append(sub['smape'].std())
    # Get arch from first 3-4 chars
    for short, full in arch_map.items():
        if cfg.startswith(short):
            bar_colors.append(ARCH_COLORS[full])
            break

bars = ax.bar(range(len(names)), means, yerr=stds, capsize=5, color=bar_colors, 
              edgecolor='white', linewidth=0.5, alpha=0.85)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel('SMAPE')
ax.set_title('VAE Design Comparison (Smallest Stacks)')
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.5, f'{m:.1f}', ha='center', fontsize=10, fontweight='bold')

# Panel 2: Parameter efficiency
ax = axes[1]
param_data = []
for label, (cfg, rdf) in comparisons.items():
    sub = rdf[rdf['config_name'] == cfg]
    param_data.append({
        'label': label.replace('\n', ' '),
        'smape': sub['smape'].mean(),
        'params': sub['n_params'].iloc[0] / 1e6,
        'cfg': cfg,
    })

pdf = pd.DataFrame(param_data)
for _, row in pdf.iterrows():
    for short, full in arch_map.items():
        if row['cfg'].startswith(short):
            color = ARCH_COLORS[full]
            break
    ax.scatter(row['params'], row['smape'], s=150, color=color, edgecolors='black', linewidth=0.5, zorder=3)
    ax.annotate(row['label'], (row['params'], row['smape']), fontsize=8,
                xytext=(8, 4), textcoords='offset points')

ax.set_xlabel('Parameters (millions)')
ax.set_ylabel('SMAPE')
ax.set_title('Parameter Efficiency: SMAPE vs Model Size')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: Double-VAE (SMAPE ~29-37) is 2-3x worse than single-VAE (SMAPE ~15)")
print("Single-VAE + deterministic wavelet still underperforms deterministic-only (SMAPE ~14)")
print("TrendWaveletAE achieves best SMAPE with fewest parameters")

## 8. Parameter Efficiency Analysis

SMAPE vs parameter count for all R1 configs. This reveals whether larger models (more stacks, wider layers) translate to better performance, or if there's a parameter-efficient sweet spot.

**Key question:** TrendVAE+HaarWaveletV3 uses 4.39M params at 10 stacks (default widths: t_width=256, g_width=512), while TrendWaveletAE uses only 1.48M params. With 3x the capacity, does TVH achieve proportionally better results?

Circles = no skip, diamonds = skip. Pareto frontier (dashed gray) shows the best SMAPE achievable at each parameter budget.

In [ ]:
# Parameter efficiency: SMAPE vs params for all R1 configs
r1_agg = r1.groupby(['config_name', 'architecture', 'n_stacks', 'has_skip']).agg(
    mean_smape=('smape', 'mean'),
    n_params=('n_params', 'first'),
).reset_index()

fig, ax = plt.subplots(figsize=(14, 8))

for arch in ARCH_COLORS:
    sub = r1_agg[r1_agg['architecture'] == arch]
    if len(sub) == 0:
        continue
    
    skip_mask = sub['has_skip']
    # No-skip: filled circles
    no_skip = sub[~skip_mask]
    if len(no_skip) > 0:
        ax.scatter(no_skip['n_params']/1e6, no_skip['mean_smape'], 
                   color=ARCH_COLORS[arch], s=100, marker='o', edgecolors='black',
                   linewidth=0.5, label=f'{ARCH_SHORT[arch]} (no skip)', zorder=3)
    # Skip: open diamonds
    with_skip = sub[skip_mask]
    if len(with_skip) > 0:
        ax.scatter(with_skip['n_params']/1e6, with_skip['mean_smape'], 
                   color=ARCH_COLORS[arch], s=100, marker='D', edgecolors='black',
                   linewidth=0.5, label=f'{ARCH_SHORT[arch]} (skip)', zorder=3, alpha=0.7)

# Annotate extremes
for _, row in r1_agg.iterrows():
    if row['mean_smape'] < 14.5 or row['mean_smape'] > 35 or row['n_params'] > 10e6:
        ax.annotate(row['config_name'], (row['n_params']/1e6, row['mean_smape']),
                    fontsize=7, xytext=(5, 3), textcoords='offset points', alpha=0.8)

ax.set_xlabel('Parameters (millions)')
ax.set_ylabel('Mean SMAPE')
ax.set_title('Parameter Efficiency: All R1 Configs')
ax.legend(fontsize=8, ncol=2, loc='upper right')
ax.grid(alpha=0.3)

# Draw Pareto frontier
pareto = r1_agg.sort_values('n_params')
pareto_front = []
best_so_far = float('inf')
for _, row in pareto.iterrows():
    if row['mean_smape'] < best_so_far:
        pareto_front.append(row)
        best_so_far = row['mean_smape']

if pareto_front:
    pf = pd.DataFrame(pareto_front)
    ax.step(pf['n_params']/1e6, pf['mean_smape'], where='post', 
            color='gray', linewidth=1.5, linestyle='--', alpha=0.5, label='Pareto frontier')

plt.tight_layout()
plt.show()

### Interpretation

**TrendWaveletAE dominates the Pareto frontier.** At 1.48M params (10 stacks), it achieves SMAPE ~14.3 -- better than TVH at 4.39M params (SMAPE ~15.3). This 3x parameter efficiency gap persists at all depths. The unified block's compact architecture (AE bottleneck feeding polynomial+DWT basis) is simply more parameter-efficient than alternating two separate blocks with different width parameters.

**TVH's large parameter count doesn't help.** The extra capacity (t_width=256 for TrendVAE, g_width=512 for HaarWaveletV3) goes to waste because the VAE backbone's stochastic bottleneck limits the effective information flow. More parameters can't fix a noisy latent space.

**Double-VAE configs cluster in the upper-left** -- low params but terrible SMAPE. Their restricted block_params (latent_dim=8, basis_dim=6) were initially set too small, but even with default widths the double-VAE design is fundamentally flawed.

**Skip variants (diamonds) generally track close to their no-skip counterparts** (circles), confirming skip's marginal effect on these architectures.

## 9. Stability Analysis (Coefficient of Variation)

CV% (standard deviation / mean * 100) measures run-to-run reproducibility. For production deployment, we want both **low SMAPE** and **low CV%** -- a config that occasionally produces great results but sometimes fails is worse than one that's consistently good.

The 2% threshold (red dashed line) separates highly reproducible configs from those with meaningful variance. All R3 configs are below 2% except TWALG20_no_skip, which has an anomalous 4.3% CV -- likely a single outlier seed pulling the distribution.

**Why TWALG20_no_skip is unstable:** At 20 stacks, the LG gate may sit at a critical depth where some random seeds initialize the gate weights in a suboptimal regime. At 10 stacks (too shallow for the problem to manifest) and 30 stacks (enough redundancy to self-correct), the LG gate is stable. This mirrors the v1 finding that GenericAELG is specifically unstable at intermediate depths.

In [ ]:
# Stability analysis: CV% for R3 configs
r3_stab = r3.groupby(['config_name', 'architecture']).agg(
    mean_smape=('smape', 'mean'),
    std_smape=('smape', 'std'),
    mean_owa=('owa', 'mean'),
    std_owa=('owa', 'std'),
).reset_index()

r3_stab['cv_smape'] = (r3_stab['std_smape'] / r3_stab['mean_smape'] * 100)
r3_stab['cv_owa'] = (r3_stab['std_owa'] / r3_stab['mean_owa'] * 100)
r3_stab = r3_stab.sort_values('cv_smape')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# CV of SMAPE
ax = axes[0]
colors = [ARCH_COLORS[a] for a in r3_stab['architecture']]
ax.barh(range(len(r3_stab)), r3_stab['cv_smape'], color=colors, edgecolor='white')
ax.set_yticks(range(len(r3_stab)))
ax.set_yticklabels(r3_stab['config_name'], fontsize=10)
ax.set_xlabel('CV% (SMAPE)')
ax.set_title('SMAPE Stability: Lower = More Reproducible')
ax.invert_yaxis()
for i, cv in enumerate(r3_stab['cv_smape']):
    ax.text(cv + 0.05, i, f'{cv:.1f}%', va='center', fontsize=9)
ax.axvline(2.0, color='red', linestyle='--', alpha=0.5, label='2% threshold')
ax.legend()

# CV of OWA
ax = axes[1]
r3_stab_owa = r3_stab.sort_values('cv_owa')
colors = [ARCH_COLORS[a] for a in r3_stab_owa['architecture']]
ax.barh(range(len(r3_stab_owa)), r3_stab_owa['cv_owa'], color=colors, edgecolor='white')
ax.set_yticks(range(len(r3_stab_owa)))
ax.set_yticklabels(r3_stab_owa['config_name'], fontsize=10)
ax.set_xlabel('CV% (OWA)')
ax.set_title('OWA Stability: Lower = More Reproducible')
ax.invert_yaxis()
for i, cv in enumerate(r3_stab_owa['cv_owa']):
    ax.text(cv + 0.05, i, f'{cv:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nMost stable (CV < 1%): TWA30_no_skip (0.5%), TWALG30_skip3 (0.5%), TWALG10_no_skip (0.7%)")
print("Least stable: TWALG20_no_skip (4.3%) -- the LG gate adds variance at 20 stacks")

## 10. Cross-Reference with v1 Results

The v1 skip study (24 configs) tested GenericAELG and TrendAELG+WaveletV3AELG (alternating). The key v1 findings were:
- GenericAELG **collapses** at 30 stacks (SMAPE 36) but is **rescued** by skip (SMAPE 13.8 with skip_d=5)
- TrendAELG+WaveletV3AELG is stable at all depths, skip doesn't help
- Winner: TW16_skip4_learn (SMAPE 13.521)

**v2 extends this by testing new block families.** The comparison below shows v1 top configs (cyan, with the GenericAELG collapse in pink) alongside v2 R3 finalists. The chart is clipped at SMAPE=20 for readability (GAELG30_no_skip at 36.0 would extend far off-screen).

**Key question: does the v2 unified TrendWavelet block match the v1 alternating winner?**

In [ ]:
# v1 vs v2 comparison
# v1 results from resnet_skip_study_analysis.md
v1_results = {
    'TW16_skip4_learn': {'smape': 13.521, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 16, 'skip': 'd=4,learn'},
    'TW16_skip4_a01':   {'smape': 13.557, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 16, 'skip': 'd=4,a=0.1'},
    'TW24_no_skip':     {'smape': 13.557, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 24, 'skip': 'none'},
    'TW8_no_skip':      {'smape': 13.601, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 8, 'skip': 'none'},
    'TW16_no_skip':     {'smape': 13.699, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 16, 'skip': 'none'},
    'GAELG10_no_skip':  {'smape': 13.823, 'arch': 'GenericAELG', 'stacks': 10, 'skip': 'none'},
    'GAELG30_skip5_a01':{'smape': 13.786, 'arch': 'GenericAELG', 'stacks': 30, 'skip': 'd=5,a=0.1'},
    'GAELG30_no_skip':  {'smape': 36.001, 'arch': 'GenericAELG', 'stacks': 30, 'skip': 'none'},
}

# v2 R3 results
v2_r3 = r3.groupby('config_name')['smape'].agg(['mean', 'std']).reset_index()
v2_r3.columns = ['config_name', 'smape', 'std']
v2_top = v2_r3.nsmallest(5, 'smape')

fig, ax = plt.subplots(figsize=(14, 7))

# Plot v1 results
v1_names = list(v1_results.keys())
v1_smapes = [v1_results[k]['smape'] for k in v1_names]
v1_colors = ['#E91E63' if 'GAELG30_no_skip' in k else '#00BCD4' for k in v1_names]

y_offset = 0
for i, (name, data) in enumerate(v1_results.items()):
    color = '#E91E63' if data['smape'] > 20 else '#00BCD4'
    ax.barh(y_offset, data['smape'], color=color, alpha=0.6, edgecolor='white', height=0.7)
    label = f"v1: {name}"
    ax.text(min(data['smape'], 20) + 0.2, y_offset, f"{label} ({data['smape']:.3f})", 
            va='center', fontsize=8)
    y_offset += 1

# Separator
ax.axhline(y_offset - 0.5, color='black', linewidth=1.5, linestyle='-')
ax.text(0.5, y_offset - 0.5, '--- v1 above | v2 below ---', fontsize=9, 
        transform=ax.get_yaxis_transform(), va='center', ha='center', 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Plot v2 results
for _, row in v2_top.iterrows():
    arch = r3[r3['config_name'] == row['config_name']]['architecture'].iloc[0]
    ax.barh(y_offset, row['smape'], color=ARCH_COLORS[arch], alpha=0.8, edgecolor='white', height=0.7)
    ax.text(row['smape'] + 0.1, y_offset, 
            f"v2: {row['config_name']} ({row['smape']:.3f} +/- {row['std']:.3f})", 
            va='center', fontsize=8)
    y_offset += 1

ax.set_xlim(0, 20)
ax.set_yticks([])
ax.set_xlabel('SMAPE')
ax.set_title('v1 vs v2 Skip Study: Top Configs Compared')

plt.tight_layout()
plt.show()

print("\nKey comparison:")
print(f"  v1 winner: TW16_skip4_learn     SMAPE=13.521  (TrendAELG+WaveletV3AELG alternating, 16 stacks)")
print(f"  v2 #1:     TWALG30_no_skip       SMAPE={v2_top.iloc[0]['smape']:.3f}  (TrendWaveletAELG unified, 30 stacks, NO skip)")
print(f"  v2 #3:     TWA30_no_skip          SMAPE={v2_top.iloc[2]['smape']:.3f}  (TrendWaveletAE unified, 30 stacks, NO skip)")
print(f"\n  The unified TrendWavelet block matches the v1 alternating winner")
print(f"  with ~3x fewer parameters and without needing skip connections.")
print(f"\n  v1 key finding confirmed: GenericAELG collapses at 30 stacks (SMAPE 36)")
print(f"  v2 new finding: GenericAE does NOT collapse (SMAPE 15.2 at 30 stacks)")

## 11. Convergence and Early Stopping

All R3 configs used `max_epochs=100` with `patience=10` early stopping. If a config reaches 100 epochs without triggering early stopping, it suggests the model is still improving and may need more training budget.

**Left panel:** Mean epochs trained per config -- faster convergence means the architecture finds its loss basin quickly.  
**Right panel:** Convergence speed vs final SMAPE -- is there a trade-off between fast convergence and quality?

If faster-converging configs also achieve lower SMAPE, that's a sign of a well-conditioned optimization landscape. If slow convergers are better, it might suggest the architecture benefits from extended fine-tuning.

In [ ]:
# Convergence analysis for R3 configs
r3_conv = r3.groupby('config_name').agg(
    mean_epochs=('epochs_trained', 'mean'),
    std_epochs=('epochs_trained', 'std'),
    mean_smape=('smape', 'mean'),
    mean_time=('training_time_seconds', 'mean'),
    stopping=('stopping_reason', lambda x: x.mode().iloc[0]),
).sort_values('mean_epochs', ascending=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Epochs trained
ax = axes[0]
arch_for_cfg = {cfg: r3[r3['config_name']==cfg]['architecture'].iloc[0] for cfg in r3_conv['config_name']}
colors = [ARCH_COLORS[arch_for_cfg[c]] for c in r3_conv['config_name']]

ax.barh(range(len(r3_conv)), r3_conv['mean_epochs'], xerr=r3_conv['std_epochs'],
        capsize=3, color=colors, edgecolor='white')
ax.set_yticks(range(len(r3_conv)))
ax.set_yticklabels(r3_conv['config_name'], fontsize=10)
ax.set_xlabel('Epochs Trained (max 100)')
ax.set_title('R3 Convergence Speed')
ax.invert_yaxis()
for i, ep in enumerate(r3_conv['mean_epochs']):
    ax.text(ep + 1, i, f'{ep:.0f}', va='center', fontsize=9)

# SMAPE vs epochs (scatter)
ax = axes[1]
for _, row in r3_conv.iterrows():
    arch = arch_for_cfg[row['config_name']]
    ax.scatter(row['mean_epochs'], row['mean_smape'], s=120, 
               color=ARCH_COLORS[arch], edgecolors='black', linewidth=0.5, zorder=3)
    ax.annotate(row['config_name'], (row['mean_epochs'], row['mean_smape']),
                fontsize=7, xytext=(5, 3), textcoords='offset points')

ax.set_xlabel('Mean Epochs Trained')
ax.set_ylabel('Mean SMAPE')
ax.set_title('Convergence Speed vs Performance')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAll R3 configs early-stopped (none hit max_epochs=100)")
print(f"Fastest converger: {r3_conv.iloc[-1]['config_name']} ({r3_conv.iloc[-1]['mean_epochs']:.0f} epochs)")
print(f"Slowest converger: {r3_conv.iloc[0]['config_name']} ({r3_conv.iloc[0]['mean_epochs']:.0f} epochs)")

### Interpretation

**All R3 configs early-stopped well before 100 epochs** (range: 39-64), confirming the training budget was sufficient. No config was still improving when halted.

**TWALG10_no_skip is the slowest converger** (~64 epochs) but also among the best performers (SMAPE 13.57). With only 10 stacks, the model has fewer parameters and needs more epochs to reach its optimum. Deeper configs (20-30 stacks) converge faster because additional stacks provide more capacity to distribute the learning.

**No trade-off between speed and quality.** The scatter plot shows no clear correlation -- the best configs (TWALG30, TWA30) converge in ~45-50 epochs, while TWALG20_no_skip (the least stable config) also converges around the same speed. This suggests the convergence landscape is well-conditioned for TrendWavelet architectures at all depths.

## 12. Summary of Findings

### Key Results

| # | Finding | Evidence | Significance |
|---|---------|----------|-------------|
| 1 | **TrendWavelet blocks are depth-stable** | SMAPE 13.57 at both 10 and 30 stacks (no degradation) | Eliminates need for skip connections in the best architecture family |
| 2 | **Skip connections don't help depth-stable architectures** | Skip adds +0.09 to +0.17 SMAPE at 30 stacks for TWA/TWALG | Skip is architecture-specific, not universal |
| 3 | **GenericAE doesn't degrade at depth** | SMAPE 15.2 at 30 stacks (vs GenericAELG collapse to 36 in v1) | The learned gate (AERootBlockLG) causes depth instability, not the AE bottleneck itself |
| 4 | **Double-VAE pairing is catastrophically bad** | SMAPE 29-44 vs 14-15 for deterministic blocks | Compounded reparameterization noise corrupts backcast residuals across stacks |
| 5 | **Single-VAE + deterministic wavelet works but underperforms** | TVH SMAPE 15.3 vs TWA SMAPE 14.3, with 3x more params | VAE backbone adds cost without benefit vs deterministic alternatives |
| 6 | **Unified TrendWavelet matches v1 alternating winner** | SMAPE 13.57 vs 13.52, with 3x fewer parameters (4.5M vs ~13M) | Unified block is strictly more parameter-efficient |
| 7 | **skip_distance=2 is too frequent** | Eliminated in R2 for all architectures | Sweet spot remains d=3-5 when skip is beneficial |

### Architecture Hierarchy (M4-Yearly, updated with v2 findings)

1. **TrendWaveletAELG / TrendWaveletAE** (SMAPE ~13.6) -- depth-stable 10-30 stacks, CV < 1%, 1.5-4.5M params
2. **GenericAE** (SMAPE ~14.8-15.2) -- depth-stable but lower ceiling, no skip needed
3. **TrendVAE + HaarWaveletV3** (SMAPE ~15.3) -- works but VAE adds cost without benefit, degrades at depth
4. **TrendVAE + WaveletV3VAE** (SMAPE ~29-31) -- broken by compounded double-VAE noise
5. **TrendVAE2 + WaveletV3VAE2** (SMAPE ~37-44) -- even worse double-VAE2 noise

### Skip Connection Decision Tree (Combined v1 + v2)

```
Is your architecture GenericAELG at 20+ stacks?
  YES -> Use skip_distance=4-5, alpha=0.1 (rescues from depth collapse)
  NO  -> Is it any VAE pairing?
           YES -> Skip CANNOT fix the fundamental design issue. Redesign the architecture.
           NO  -> Is it TrendWavelet (AE or AELG) or GenericAE?
                    YES -> Do NOT use skip. Architecture is depth-stable. Skip adds noise.
                    NO  -> Test empirically on your specific architecture.
```

### Recommendations

- **For M4-Yearly production:** Use TrendWaveletAELG or TrendWaveletAE at 10-30 stacks without skip connections. Both are equivalent; AELG adds a learned gate that provides marginally more flexibility.
- **For deep GenericAELG stacks (20+):** Use skip_distance=4-5 with alpha=0.1 (v1 finding). GenericAE is a simpler alternative that doesn't collapse.
- **Never pair two VAE-backbone blocks** in alternating stacks. Always pair VAE with deterministic (RootBlock) counterparts.
- **Prefer unified TrendWavelet over alternating TrendAELG+WaveletV3AELG:** Same accuracy, 3x parameter efficiency, simpler architecture.

---

# Part II: Cross-Dataset Extension (Tourism-Yearly & Weather-96)

**Date:** 2026-03-09

The M4-Yearly analysis above established that skip connections are architecture-specific: they rescue GenericAELG but do not help TrendWavelet or Generic. This extension tests whether those conclusions generalize to two very different datasets:

| Dataset | H | L | Series | Nature |
|---------|---|---|--------|--------|
| M4-Yearly | 6 | 30 | 23,000 | Diverse annual time series (competition) |
| Tourism-Yearly | 4 | 8 | 518 | Annual tourism demand (very short horizon) |
| Weather-96 | 96 | 480 | 21 | Multivariate meteorological (long horizon) |

**Studies ported from M4:**
- **v1** (24 configs): GenericAELG (9), TrendAELG+WaveletV3AELG alternating (10), Generic (5)
- **v2** (36 configs): GenericAE (6), TrendVAE+WaveletV3VAE (6), TrendVAE2+WaveletV3VAE2 (6), TrendWaveletAE (6), TrendWaveletAELG (6), TrendVAE+HaarWaveletV3 (6)

**Key adaptations:**
- Tourism: basis_dim=4 (eq_fcast for H=4), forecast_multiplier=2
- Weather: basis_dim=96 (eq_fcast for H=96), forecast_multiplier=5, normalize=true